# Проверка гипотезы о равенстве математических ожиданий. Выполнил Григорьев Даниил Александрович, поток 21.1, ИСУ: 465635
**Вариант 2**

Проверяется гипотеза о равенстве математических ожиданий двух независимых выборок из нормальных совокупностей при неизвестных, но равных дисперсиях.

Используется двусторонний критерий Стьюдента для двух независимых выборок с объединённой оценкой дисперсии


In [20]:
import pandas as pd
import numpy as np
from scipy import stats


# Данные из файлов variant_2_sample_X.csv и variant_2_sample_Y.csv
X = np.array(pd.read_csv('variant_2_sample_X.csv')["x_i"])
Y = np.array(pd.read_csv('variant_2_sample_Y.csv')["y_j"])


x_df = pd.DataFrame({"i": range(1, len(X)+1), "x_i": X})
y_df = pd.DataFrame({"i": range(1, len(Y)+1), "y_j": Y})

display(x_df.head())
display(y_df.head())

,i,x_i
0,1,24
1,2,22
2,3,24
3,4,24
4,5,19


,i,y_j
0,1,21
1,2,24
2,3,25
3,4,20
4,5,31


## 1. Постановка задачи

Дано:

- $H_0: \mu_1 = \mu_2$
- $H_1: \mu_1 \ne \mu_2$
- уровень значимости $\alpha = 0.05$
- объёмы выборок: $m = 44$, $n = 48$

По условию считаем, что:

1. выборки независимы;
2. генеральные совокупности нормальны;
3. дисперсии неизвестны, но равны.

Тогда статистика критерия:
$$
T = \frac{\overline{X} - \overline{Y}}
{s_p \sqrt{\frac{1}{m} + \frac{1}{n}}},
$$
где объединённая оценка дисперсии:
$$
s_p^2 = \frac{(m-1)s_X^2 + (n-1)s_Y^2}{m+n-2}.
$$

При справедливости $H_0$:
$$
T \sim t(m+n-2).
$$


In [21]:
# Основные выборочные характеристики
m = len(X)
n = len(Y)
alpha = 0.05

x_mean = X.mean()
y_mean = Y.mean()

s2_x = X.var(ddof=1)
s2_y = Y.var(ddof=1)

summary = pd.DataFrame({
    "Показатель": ["Объём выборки", "Среднее", "Несмещённая дисперсия", "Стандартное отклонение"],
    "X": [m, x_mean, s2_x, np.sqrt(s2_x)],
    "Y": [n, y_mean, s2_y, np.sqrt(s2_y)]
})

summary

,Показатель,X,Y
0,Объём выборки,44.000000,48.000000
1,Среднее,20.022727,21.750000
2,Несмещённая дисперсия,13.836681,13.382979
3,Стандартное отклонение,3.719769,3.658275


## 2. Расчёт статистики критерия вручную

In [22]:
# Объединённая оценка дисперсии
s2_p = ((m - 1) * s2_x + (n - 1) * s2_y) / (m + n - 2)
s_p = np.sqrt(s2_p)

# Наблюдаемое значение t-статистики
t_obs = (x_mean - y_mean) / (s_p * np.sqrt(1/m + 1/n))

# Число степеней свободы
df = m + n - 2

# Критическое значение для двустороннего критерия
t_crit = stats.t.ppf(1 - alpha/2, df)

# p-value
p_value = 2 * (1 - stats.t.cdf(abs(t_obs), df))

results = pd.DataFrame({
    "Величина": [
        "Среднее X̄", "Среднее Ȳ", "s²_X", "s²_Y", "s²_p", "s_p",
        "t_набл", "t_крит", "df", "p-value"
    ],
    "Значение": [
        x_mean, y_mean, s2_x, s2_y, s2_p, s_p,
        t_obs, t_crit, df, p_value
    ]
})

results

,Величина,Значение
0,Среднее X̄,20.022727
1,Среднее Ȳ,21.750000
2,s²_X,13.836681
3,s²_Y,13.382979
4,s²_p,13.599747
5,s_p,3.687784
6,t_набл,-2.244133
7,t_крит,1.986675
8,df,90.000000
9,p-value,0.027276


## 3. Правило принятия решения

Для двустороннего критерия область принятия нулевой гипотезы:
$$
|t_{\text{набл}}| \le t_{\text{крит}}.
$$

Иначе нулевая гипотеза отвергается.


In [23]:
decision_manual = "Отвергаем H0" if abs(t_obs) > t_crit else "Нет оснований отвергнуть H0"
decision_p = "Отвергаем H0" if p_value < alpha else "Нет оснований отвергнуть H0"

print(f"|t_набл| = {abs(t_obs):.4f}")
print(f"t_крит = {t_crit:.4f}")
print(f"p-value = {p_value:.4f}")
print(f"Решение по критической области: {decision_manual}")
print(f"Решение по p-value: {decision_p}")

|t_набл| = 2.2441
t_крит = 1.9867
p-value = 0.0273
Решение по критической области: Отвергаем H0
Решение по p-value: Отвергаем H0


## 4. Проверка с помощью `scipy.stats.ttest_ind`

In [24]:
ttest = stats.ttest_ind(X, Y, equal_var=True)
ttest

TtestResult(statistic=np.float64(-2.2441327148203998), pvalue=np.float64(0.027276097472721337), df=np.float64(90.0))

## 5. Итоговый вывод

По данным выборок получено:

- $\overline{X} \approx 20.0227$
- $\overline{Y} \approx 21.7500$
- $t_{\text{набл}} \approx -2.2441$
- $t_{\text{крит}} \approx 1.9867$
- $p$-value $\approx 0.0273$

Так как:

- $|t_{\text{набл}}| > t_{\text{крит}}$
- и одновременно $p$-value $< 0.05$,

нулевая гипотеза $H_0$ отвергается

### Заключение
На уровне значимости 0.05 имеются статистически значимые основания считать, что математические ожидания двух генеральных совокупностей различаются


## 6. Ответ

Для двух независимых выборок из нормальных совокупностей при неизвестных, но равных дисперсиях применён двусторонний критерий Стьюдента с объединённой оценкой дисперсии.

Получено:
- $\overline{X} = 20.0227$
- $\overline{Y} = 21.7500$
- $s_X^2 = 13.8367$
- $s_Y^2 = 13.3830$
- $s_p^2 = 13.5997$
- $t_{набл} = -2.2441$
- $t_{крит} = 1.9867$, $df = 90$

Так как $|t_{набл}| = 2.2441 > 1.9867$, нулевая гипотеза $H_0$ отвергается.

**Ответ:** на уровне значимости $\alpha = 0.05$ средние значения в двух совокупностях статистически различаются
